In [8]:
import pandas as pd
import numpy as np

In [9]:
# Read input data from FRBNY website
col_gdp =  pd.read_excel("inputData/col_data.xlsx", sheet_name="GDP")
col_gdp = col_gdp.set_index("Date")
col_gdp['gdp.log'] = np.log(col_gdp['GDP'])
#drop gdp raw 
col_gdp = col_gdp.drop(columns = ['GDP'])
#convert index to datetime(e.g. 3/31/2025)
col_gdp.index = pd.to_datetime(col_gdp.index,)

col_gdp = col_gdp.resample("QS").last()
col_gdp





C:\Users\tfarotimi\AppData\Local\Temp\ipykernel_32072\2608156582.py:8: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  col_gdp.index = pd.to_datetime(col_gdp.index,)


,gdp.log
Date,
2005-01-01,11.744741
2005-04-01,11.766842
2005-07-01,11.764554
2005-10-01,11.784829
2006-01-01,11.807113
...,...
2022-04-01,12.405851
2022-07-01,12.414408
2022-10-01,12.397663


In [10]:
col_inflation =  pd.read_excel("inputData/col_data.xlsx", sheet_name="inflation")
col_inflation = col_inflation.set_index("Date")

#resample 'index'column to quarterly
col_inflation = col_inflation['inflation'].resample('QS').mean()
col_inflation['growth'] = col_inflation.pct_change() * 100
col_inflation['growth'] = col_inflation['growth'].dropna()

#annualize quarterly growth
col_inflation['annualized'] =  100 * ((1 + col_inflation['growth']/100)**4 - 1)
col_inflation['annualized']
col_inflation_final = col_inflation['annualized'].to_frame()
col_inflation_final['expectations'] = col_inflation_final.rolling(4).mean()
col_inflation_final = col_inflation_final.dropna()

col_inflation_final

,inflation,expectations
Date,,
2000-01-01,12.106535,7.604037
2000-04-01,10.679337,7.674292
2000-07-01,4.665784,7.924477
2000-10-01,3.840676,7.823083
2001-01-01,10.853333,7.509782
...,...,...
2024-01-01,10.349797,8.152799
2024-04-01,7.257125,7.215509
2024-07-01,2.627392,6.428299


In [11]:
#.interest    
col_interest =  pd.read_excel("inputData/col_data.xlsx", sheet_name="interest")
col_interest = col_interest.set_index("Date")
col_interest = col_interest.resample('QS').mean()
col_interest['interest'] = col_interest['interest'].dropna()
col_interest

,interest
Date,
1995-01-01,22.648855
1995-04-01,25.059111
1995-07-01,20.683983
1995-10-01,26.337160
1996-01-01,29.828766
...,...
2024-04-01,11.923870
2024-07-01,10.927486
2024-10-01,9.919207


In [20]:
#merge col_gdp, col_inflation_final, col_interest
col_data = pd.concat([col_gdp, col_inflation_final, col_interest], axis=1).dropna()
col_data.index = col_data.index.to_period('Q')

In [13]:
stringency = pd.read_csv("inputData/stringency.csv")
stringency_data = stringency[['CountryCode', 'Date', 'StringencyIndex_Average']]
stringency_data = stringency_data[stringency_data['CountryCode'] == 'COL']
stringency_data = stringency_data.drop(columns=['CountryCode'])
# stringency_data['Date'] = pd.to_datetime(stringency_data['Date'])
stringency_data = stringency_data.set_index('Date')
#process 20200101 to 2020-01-01
stringency_data.index = pd.to_datetime(stringency_data.index, format='%Y%m%d')
stringency_data = stringency_data.resample('QS').mean()
stringency_data = stringency_data.rename(columns={'StringencyIndex_Average': 'covid.indicator'})
stringency_data.index = stringency_data.index.to_period('Q')
#add rows up to 2024Q1 with stringency declining linearly to 0
stringency_data = stringency_data.reindex(pd.period_range(start='2020Q1', end='2024Q1', freq='Q', name='Date'))
stringency_data.loc['2024Q1'] = 0
stringency_data = stringency_data.interpolate(method='linear')
#add rows forom 1990Q1 to 2019Q4 with stringency = 0
stringency_data = stringency_data.reindex(pd.period_range(start='1990Q1', end='2024Q1', freq='Q', name='Date'))
stringency_data = stringency_data.fillna(0)
stringency_data = stringency_data.dropna()
stringency_data

,covid.indicator
Date,
1990Q1,0.000
1990Q2,0.000
1990Q3,0.000
1990Q4,0.000
1991Q1,0.000
...,...
2023Q1,15.552
2023Q2,11.664
2023Q3,7.776


In [21]:
col_data = col_data.merge(stringency_data, left_index=True, right_on='Date')


In [22]:
col_data

,gdp.log,inflation,expectations,interest,covid.indicator
Date,,,,,
2005Q1,11.744741,7.918564,5.362321,6.405073,0.000000
2005Q2,11.766842,4.597563,4.810320,6.371915,0.000000
2005Q3,11.764554,2.212565,4.340969,6.259546,0.000000
2005Q4,11.784829,1.629163,4.089464,5.702026,0.000000
2006Q1,11.807113,5.331714,3.442751,5.922055,0.000000
...,...,...,...,...,...
2022Q2,12.405851,11.957166,8.127023,5.757249,26.613516
2022Q3,12.414408,9.960037,9.494363,8.576998,20.317174
2022Q4,12.397663,9.712111,11.169197,10.822290,19.440000


In [23]:
#rename columns
col_data.columns = ['gdp.log', 'inflation', 'inflation.expectations', 'interest', 'covid.indicator']

#add columns of 0 called covid.indicator
#col_data['covid.indicator'] = 0

#save data to excel
col_data.to_excel("inputData/Holston_Laubach_Williams_COL.xlsx", sheet_name="COL Input Data" )
